<a href="https://colab.research.google.com/github/CPTR295/NLP-Using-Transformers/blob/main/MultiLingual_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datasets import get_dataset_config_names
xtreme_subsets = get_dataset_config_names("google/xtreme")
print(f"XTREME Subsets:{ len(xtreme_subsets)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


XTREME Subsets:183


In [2]:
panx_sunbsets = [s for s in xtreme_subsets if s.startswith("PAN")]
print(f"PAN Subsets:{len(panx_sunbsets)}")

PAN Subsets:40


In [3]:
from datasets import load_dataset
load_dataset("google/xtreme", name="PAN-X.de")

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs'],
        num_rows: 10000
    })
})

In [4]:
from collections import defaultdict
from datasets import DatasetDict

langs = ['de','fr','it','en']
fracs = [0.629, 0.229, 0.084, 0.059]
panx_ch = defaultdict(DatasetDict)

for lang,frac in zip(langs,fracs):
  ds = load_dataset("google/xtreme",name=f'PAN-X.{lang}')
  for split in ds:
    panx_ch[lang][split] = (
        ds[split].shuffle(seed=0).select(range(int(frac*ds[split].num_rows)))
    )

In [5]:
import pandas as pd
pd.DataFrame({lang:[panx_ch[lang]['train'].num_rows] for lang in langs},index=['Number of training samples'])

,de,fr,it,en
Number of training samples,12580,4580,1680,1180


In [6]:
element = panx_ch['de']['train'][0]
for k,v in element.items():
  print(f'{k}:{v}')

tokens:['2.000', 'Einwohnern', 'an', 'der', 'Danziger', 'Bucht', 'in', 'der', 'polnischen', 'Woiwodschaft', 'Pommern', '.']
ner_tags:[0, 0, 0, 0, 5, 6, 0, 0, 5, 5, 6, 0]
langs:['de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de', 'de']


In [7]:
for k,v in panx_ch['de']['train'].features.items():
  print(f'{k}:{v}')

tokens:List(Value('string'))
ner_tags:List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))
langs:List(Value('string'))


In [8]:
tags = panx_ch['de']['train'].features['ner_tags'].feature
print(tags)

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'])


In [9]:
def create_tag_names(batch):
  return {'ner_tags_str':[tags.int2str(idx) for idx in batch['ner_tags']]}
panx_de = panx_ch['de'].map(create_tag_names)

In [10]:
de_example = panx_de['train'][0]
pd.DataFrame([de_example['tokens'],de_example['ner_tags_str']],index=['Tokens','Tags'])

,0,1,2,3,4,5,6,7,8,9,10,11
Tokens,2.000,Einwohnern,an,der,Danziger,Bucht,in,der,polnischen,Woiwodschaft,Pommern,.
Tags,O,O,O,O,B-LOC,I-LOC,O,O,B-LOC,B-LOC,I-LOC,O


In [11]:
from collections import Counter
split2freqs = defaultdict(Counter)
for split,dataset in panx_de.items():
  for row in dataset['ner_tags_str']:
    for tag in row:
      if tag.startswith('B'):
        tag_type = tag.split('-')[1]
        split2freqs[split][tag_type] += 1
pd.DataFrame.from_dict(split2freqs,orient='index')

,LOC,ORG,PER
train,6186,5366,5810
validation,3172,2683,2893
test,3180,2573,3071


Tokenization

In [12]:
from transformers import AutoTokenizer
bert_model_name = 'bert-base-cased'
xlmr_model_name = 'xlm-roberta-base'
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
xlmr_tokenizer = AutoTokenizer.from_pretrained(xlmr_model_name)

In [13]:
text = 'Jack Sparrow Loves pirates'
bert_tokens = bert_tokenizer.tokenize(text)
xlmr_tokens = xlmr_tokenizer.tokenize(text)
print(f'BERT Tokens:{bert_tokens}')
print(f'XLM-R Tokens:{xlmr_tokens}')

BERT Tokens:['Jack', 'Spa', '##rrow', 'Loves', 'pirates']
XLM-R Tokens:['▁Jack', '▁Spar', 'row', '▁Love', 's', '▁pirat', 'es']


Custom Model for Token Classification

In [14]:
import torch.nn as nn
from transformers import XLMRobertaConfig
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [25]:
class XLMRobertaForTokenClassification(RobertaPreTrainedModel):
  config_class = XLMRobertaConfig
  def __init__(self,config):
    super().__init__(config)
    self.num_labels = config.num_labels
    self.roberta = RobertaModel(config,add_pooling_layer=False)
    self.dropout = nn.Dropout(config.hidden_dropout_prob)
    self.classifier = nn.Linear(config.hidden_size,config.num_labels)
    self.init_weights()

  def forward(self,input_ids=None,attention_mask=None,token_type_ids=None,labels = None,**kwargs):
    outputs = self.roberta(input_ids,attention_mask=attention_mask,token_type_ids=token_type_ids,**kwargs)
    sequence_output = self.dropout(outputs[0])
    logits = self.classifier(sequence_output)
    loss = None
    if labels is not None:
      loss_fct = nn.CrossEntropyLoss()
      loss = loss_fct(logits.view(-1,self.num_labels),labels.view(-1))
    return TokenClassifierOutput(loss=loss,logits=logits,hidden_states=outputs.hidden_states,attentions=outputs.attentions)

In [16]:
index2tag = {idx:tag for idx,tag in enumerate(tags.names)}
tag2index = {tag:idx for idx,tag in enumerate(tags.names)}

In [17]:
from transformers import AutoConfig
xlmr_config = AutoConfig.from_pretrained(xlmr_model_name,num_labels=tags.num_classes,id2label=index2tag,label2id=tag2index)

In [18]:
!pip uninstall -y transformers
!pip install transformers==4.57.1

Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)


In [19]:
import transformers
import torch

print(transformers.__version__)
print(torch.__version__)
print(xlmr_model_name)
print(xlmr_config)

4.57.1
2.9.0+cpu
xlm-roberta-base
XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-PER",
    "2": "I-PER",
    "3": "B-ORG",
    "4": "I-ORG",
    "5": "B-LOC",
    "6": "I-LOC"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "B-LOC": 5,
    "B-ORG": 3,
    "B-PER": 1,
    "I-LOC": 6,
    "I-ORG": 4,
    "I-PER": 2,
    "O": 0
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "output_past": true,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.57.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 250002
}



In [26]:
import torch
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import torch_xla
import torch_xla.core.xla_model as xm

device = xm.xla_device()
xlmr_model = XLMRobertaForTokenClassification.from_pretrained(xlmr_model_name,config=xlmr_config).to(device)

/tmp/ipykernel_3084/760576513.py:6: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
input_ids = xlmr_tokenizer.encode(text,return_tensors='pt')
pd.DataFrame([xlmr_tokens,input_ids[0].numpy()],index=['Tokens','Input IDs'])

,0,1,2,3,4,5,6,7,8
Tokens,▁Jack,▁Spar,row,▁Love,s,▁pirat,es,NaN,NaN
Input IDs,0,21763,37456,15555,11954,7,78312,90.0,2.0


In [28]:
outputs = xlmr_model(input_ids.to(device)).logits
predictions = torch.argmax(outputs,dim=-1)
print(f"Number of tokens in sequence: {len(xlmr_tokens)}")
print(f"Shape of outputs: {outputs.shape}")

Number of tokens in sequence: 7
Shape of outputs: torch.Size([1, 9, 7])


In [29]:
preds = [tags.names[p] for p in predictions[0].cpu().numpy()]
pd.DataFrame([xlmr_tokens, preds], index=["Tokens", "Tags"])

,0,1,2,3,4,5,6,7,8
Tokens,▁Jack,▁Spar,row,▁Love,s,▁pirat,es,None,None
Tags,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER,I-PER


In [30]:
def tag_text(text,tags,model,tokenizer):
  tokens = tokenizer(text)
  input_ids = xlmr_tokenizer(text,return_tensors='pt').input_ids.to(device)
  outputs = model(input_ids)[0]
  predictions = torch.argmax(outputs,dim=-1)
  preds = [tags.names[p] for p in predictions[0].cpu().numpy()]
  return pd.DataFrame([tokens,preds],index=['Tokens','Tags'])

In [31]:
words,labels = de_example['tokens'],de_example['ner_tags']

In [32]:
tokenized_input = xlmr_tokenizer(de_example['tokens'],is_split_into_words=True)
tokens = xlmr_tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'])
pd.DataFrame([tokens,tokenized_input['input_ids']],index=['Tokens','Input IDs'])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Input IDs,0,70101,176581,19,142,122,2290,708,1505,18363,...,13787,14,15263,18917,663,6947,19,6,5,2


In [34]:
word_ids = tokenized_input.word_ids()
pd.DataFrame([tokens,word_ids],index=['Tokens','Word IDs'])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None


In [35]:
previous_word_idx = None
label_ids = []
for word_idx in word_ids:
  if word_idx is None or word_idx == previous_word_idx:
    label_ids.append(-100)
  elif word_idx != previous_word_idx:
    label_ids.append(labels[word_idx])
  previous_word_idx = word_idx
labels = [index2tag[l] if l!=-100 else 'IGN' for l in label_ids]
index = ['Tokens','Word IDs','Label IDs','Labels']
pd.DataFrame([tokens,word_ids,label_ids,labels],index=index)

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
Tokens,<s>,▁2.000,▁Einwohner,n,▁an,▁der,▁Dan,zi,ger,▁Buch,...,▁Wo,i,wod,schaft,▁Po,mmer,n,▁,.,</s>
Word IDs,None,0,1,1,2,3,4,4,4,5,...,9,9,9,9,10,10,10,11,11,None
Label IDs,-100,0,0,-100,0,0,5,-100,-100,6,...,5,-100,-100,-100,6,-100,-100,0,-100,-100
Labels,IGN,O,O,IGN,O,O,B-LOC,IGN,IGN,I-LOC,...,B-LOC,IGN,IGN,IGN,I-LOC,IGN,IGN,O,IGN,IGN


In [46]:
def toeknize_and_align_labels(examples):
  tokenized_inputs = xlmr_tokenizer(examples['tokens'],truncation=True,is_split_into_words=True)
  labels = []
  for idx,label in enumerate(examples['ner_tags']):
    word_ids = tokenized_inputs.word_ids(batch_index=idx)
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
      if word_idx is None or word_idx == previous_word_idx:
        label_ids.append(-100)
      elif word_idx != previous_word_idx:
        label_ids.append(label[word_idx])
      previous_word_idx = word_idx
    labels.append(label_ids)
  tokenized_inputs['labels'] = labels
  return tokenized_inputs

In [47]:
def encode_panx_dataset(corpus):
  return corpus.map(toeknize_and_align_labels,batched=True,remove_columns=['langs','ner_tags','tokens'])

In [48]:
panx_de_encoded = encode_panx_dataset(panx_ch['de'])

Map:   0%|          | 0/12580 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

Map:   0%|          | 0/6290 [00:00<?, ? examples/s]

In [49]:
import numpy as np
def align_predictions(predictions,label_ids):
  preds = np.argmax(predictions,axis=2)
  batch_size ,seq_len = preds.shape
  labels_list,preds_list =[],[]
  for batch_idx in range(batch_size):
    example_labels,example_preds = [],[]
    for seq_idx in range(seq_len):
      if label_ids[batch_idx,seq_idx] != -100:
        example_labels.append(index2tag[label_ids[batch_idx][seq_idx]])
        example_preds.append(index2tag[preds[batch_idx][seq_idx]])
    labels_list.append(example_labels)
    preds_list.append(example_preds)
  return preds_list,labels_list

Fine-Tuning XLM-RoBERTa

In [62]:
from transformers import TrainingArguments
num_epochs = 3
batch_size = 24
logging_steps = len(panx_de_encoded['train'])//batch_size
model_name = f'{xlmr_model_name}-finetuned-panx-de'
training_args = TrainingArguments(
   output_dir = model_name,log_level='error',num_train_epochs=num_epochs,per_device_train_batch_size=batch_size,
   per_device_eval_batch_size=batch_size,eval_strategy='epoch',save_steps=1e6,weight_decay=0.01,disable_tqdm=False,
   logging_steps=logging_steps,push_to_hub=False,optim="adamw_torch")

In [51]:
from seqeval.metrics import f1_score
def compute_metrics(eval_pred):
  y_pred,y_true = align_predictions(eval_pred.predictions,eval_pred.label_ids)
  return {'f1_score':f1_score(y_true,y_pred)}

In [52]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=xlmr_tokenizer)

In [53]:
def model_init():
  return (XLMRobertaForTokenClassification.from_pretrained(xlmr_model_name,config=xlmr_config).to(device))

In [54]:
%env TOKENIZERS_PARALLELISM=false

env: TOKENIZERS_PARALLELISM=false


In [56]:
from transformers import Trainer
trainer = Trainer(model_init=model_init,args=training_args,data_collator=data_collator,compute_metrics=compute_metrics,train_dataset=panx_de_encoded['train'],eval_dataset=panx_de_encoded['validation'],processing_class=xlmr_tokenizer)

In [57]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TypeError: RobertaModel.forward() got an unexpected keyword argument 'num_items_in_batch'

In [58]:
class CustomTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        outputs = model(**inputs)

        loss = outputs.loss

        return (loss, outputs) if return_outputs else loss

In [70]:
trainer = CustomTrainer(model_init=model_init,args=training_args,data_collator=data_collator,compute_metrics=compute_metrics,train_dataset=panx_de_encoded['train'],eval_dataset=panx_de_encoded['validation'],processing_class=xlmr_tokenizer)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss


In [61]:
device #for tpu add optim="adamw_torch" to traingArguments

device(type='xla', index=0)

In [ ]:
df = pd.DataFrame(trainer.state.log_history)[['epoch','loss' ,'eval_loss', 'eval_f1']]
df = df.rename(columns={"epoch":"Epoch","loss": "Training Loss", "eval_loss": "Validation Loss", "eval_f1":"F1"})
df['Epoch'] = df["Epoch"].apply(lambda x: round(x))
df['Training Loss'] = df["Training Loss"].ffill()
df[['Validation Loss', 'F1']] = df[['Validation Loss', 'F1']].bfill().ffill()
df.drop_duplicates()

In [ ]:
text_de = "Jeff Dean ist ein Informatiker bei Google in Kalifornien"
tag_text(text_de, tags, trainer.model, xlmr_tokenizer)